# Week 2 — Return Prediction

FIN 43800 · Machine Learning in Finance

Andreas Neuhierl

# Where We Are

## The Course So Far, and Next

| Meetings | Topic                                              |
|:---------|:---------------------------------------------------|
| 1–2      | Foundations: Python, returns, in and out of sample |
| **3–4**  | **Return prediction** ← this week                  |
| 5–6      | Ridge, lasso, cross-validation                     |
| 7–8      | Trees, forests, neural networks                    |
| 9–10     | Factor models                                      |
| 11–12    | Portfolio construction                             |
| 13–14    | Text, then project presentations                   |

- **This week:** can anything forecast the stock market, and would it
  pay? Then: many predictors at once, where machine learning starts.

## Two Kinds of Forecast

Picture a table of returns: one row per month, one column per stock.

- **Time series:** one column, one asset over time. *When* should I hold
  more stocks? This week and the first problem sets.
- **Cross-section:** one row, many assets in one month. *Which* stocks
  should I hold? Later in the course.

## Time Series and Cross-Section

- **Time series:** beat the **historical mean**.  
  About 100 years, one number a year (or 1,200 months).
- **Cross-section:** beat “**every stock earns the same**”.  
  Thousands of stocks, every month: far more data for machine learning.

## Return Forecasting Is Everywhere

| Market             | Forecast it with       | The catch                 |
|:-------------------|:-----------------------|:--------------------------|
| Stocks (this week) | dividends / price      | noisy, slow predictor     |
| Government bonds   | long − short yield     | predictors move together  |
| Corporate bonds    | BAA − AAA yield        | rare trades, stale prices |
| Currencies         | rate here − abroad     | pays, then crashes        |
| Commodities        | futures vs. spot price | contracts must be rolled  |
| Real estate        | rent / price           | smoothed appraisals       |
| Options            | implied − realized vol | short, lopsided histories |

## Check: Which One Is a Cross-Sectional Forecast?

- **Fingers up:** which of these forecasts is cross-sectional?
  1.  Next year’s S&P 500 return minus the T-bill return
  2.  Next month’s T-bill rate
  3.  Which of 500 stocks will beat the others next month
  4.  Next year’s inflation

## Upload, Then Run Setup

Starting a new Colab session? Colab deletes uploaded files when a
session ends. Download `gw_monthly.csv` and `gw_annual.csv` from
Brightspace, drag them into the Files pane (folder icon on the left),
then run the **Setup** cell below before any other cell.

## Setup: What It Builds

- Reads the two files into **`monthly`** (one row per month, 1871–2025)
  and **`annual`** (one row per year, 1871–2024).
- Adds three columns to `annual`:
  - `log_dp`: log D/P at the end of each year.
  - `log_dp_last_year`: one row down, known before the year starts.
    **The predictor.**
  - `div_growth`: dividend growth. **A second thing to forecast.**

<!-- -->

- **Your Turn** code is text in this notebook: click **+ Code**, paste
  it, fill the blank, then Shift + Enter.

Run this first, every session. It is week 1’s code, plus one new line:
`div_growth`.

In [ ]:
import sys
from pathlib import Path
IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    # Fetch the one helper file we need from the course repo.
    import subprocess
    subprocess.run(
        ["wget", "-q", "-O", "mlfin_data.py",
         "https://raw.githubusercontent.com/aneuhierl/ml-in-finance-fall2026/main/src/mlfin_data.py"],
        check=True,
    )
else:
    # Find src/mlfin_data.py by walking UP from this notebook's folder.
    here = Path.cwd().resolve()
    src_folders = [folder / "src" for folder in [here, *here.parents]]
    matches = [folder for folder in src_folders if (folder / "mlfin_data.py").is_file()]
    sys.path.insert(0, str(matches[0]) if matches else "../src")
from mlfin_data import resolve_data_path

import numpy as np                 # arrays of numbers
import pandas as pd                # tables with columns
import matplotlib.pyplot as plt    # charts
import statsmodels.api as sm       # regression tables

monthly = pd.read_csv(resolve_data_path("gw_monthly.csv"))
monthly.index = pd.to_datetime(monthly["yyyymm"].astype(str), format="%Y%m")
annual = pd.read_csv(resolve_data_path("gw_annual.csv")).set_index("yyyy")
# Returns are SIMPLE, in decimals per period (0.05 = 5%): per month in monthly, per
# year in annual, never annualized. equity = stocks minus T-bills.
# D/P = 12-month dividends / price. annual has no price: use each December.
december = monthly[monthly.index.month == 12]
log_dp = np.log(december["d12"]) - np.log(december["Index"])   # ln(D/P)
log_dp.index = december.index.year     # label the rows by year, like annual
annual["log_dp"] = log_dp              # pandas lines the rows up by year
annual["log_dp_last_year"] = annual["log_dp"].shift(1)
# Dividend growth, SIMPLE: this year's 12-month dividends over last year's, minus 1
annual["div_growth"] = annual["d12"] / annual["d12"].shift(1) - 1
plt.rcParams["font.size"] = 18         # big text for the projector
print(monthly.shape, annual.shape)

# Stocks Beat Bills, by a Lot and Noisily

## One Dollar in 1925

- \$1 put in US stocks at the end of 1925, dividends reinvested, after
  inflation.
- **Predict first:** what is it worth at the end of 2024?
  1.  \$15
  2.  \$90
  3.  \$900
  4.  \$9,000

## One Dollar in 1925: The Answer

- Log scale: equal vertical steps are equal **percent** growth. From 1
  to 10 is as tall as from 100 to 1,000.

In [ ]:
# REAL wealth: each year, (1 + return) / (1 + inflation). Simple returns compound.
# ret: S&P 500 return, dividends reinvested. Rfree: T-bill return. infl: CPI inflation.
stock_wealth = 1.00                 # $1 at the end of 1925
bill_wealth = 1.00
stock_path = []
bill_path = []
for year in range(1926, 2025):      # 1926, 1927, ..., 2024: the end is left out
    stock_wealth = stock_wealth * (1 + annual.loc[year, "ret"]) / (1 + annual.loc[year, "infl"])
    bill_wealth = bill_wealth * (1 + annual.loc[year, "Rfree"]) / (1 + annual.loc[year, "infl"])
    stock_path.append(stock_wealth)
    bill_path.append(bill_wealth)

plt.figure(figsize=(10, 3.8))
plt.plot(range(1926, 2025), stock_path, color="tab:blue", linewidth=3)
plt.plot(range(1926, 2025), bill_path, color="tab:gray", linewidth=3)
plt.yscale("log")                   # equal steps = equal PERCENT growth
plt.yticks([1, 10, 100, 1000], ["1", "10", "100", "1,000"])
plt.minorticks_off()
plt.text(1930, 200, f"stocks: ${stock_wealth:,.0f}", color="tab:blue")
plt.text(1985, 3, f"T-bills: ${bill_wealth:.2f}", color="tab:gray")
plt.xlim(1925, 2025)
plt.title("$1 from the end of 1925, after inflation")
plt.ylabel("real wealth (log scale)")
plt.show()

## The Premium Is Big, and Noisy

- Week 1: monthly, the premium’s standard deviation is several times its
  mean. Now year by year:

In [ ]:
# equity = stocks minus T-bills: SIMPLE, in decimals, per year. Plotted in percent.
annual_premium = annual.loc[1926:2024, "equity"]
# True counts as 1 and False as 0, so the mean of a condition is a share
share_lost = (annual_premium < 0).mean()
worst_year = annual_premium.idxmin()
best_year = annual_premium.idxmax()
bar_colors = []
for value in annual_premium:
    if value < 0:
        bar_colors.append("tab:orange")     # a year stocks lost to bills
    else:
        bar_colors.append("tab:blue")
plt.figure(figsize=(10, 3.4))
plt.bar(annual_premium.index, 100 * annual_premium, color=bar_colors)
plt.axhline(100 * annual_premium.mean(), color="black", linestyle="--")
plt.text(worst_year + 2, 100 * annual_premium[worst_year],
         f"{worst_year}: {annual_premium[worst_year]:.0%}", va="center")
plt.text(best_year + 2, 100 * annual_premium[best_year],
         f"{best_year}: +{annual_premium[best_year]:.0%}", va="center")
plt.ylim(-70, 90)
plt.text(1927, 75, "blue: stocks won   orange: stocks lost   dashed: mean", fontsize=16)
plt.title(f"Mean {annual_premium.mean():.1%}, sd {annual_premium.std():.1%}; "
          f"stocks lost in {share_lost:.0%} of years")
plt.ylabel("stocks minus bills (%)")
plt.show()

## Forecasts Set the Stock Weight

Mean-variance utility, with weight $w$ in stocks and $1 - w$ in T-bills:

$$ U(w) = r_f + w\,\mu - \tfrac{\gamma}{2}\, w^2 \sigma^2
\quad\Longrightarrow\quad
w^* = \frac{\mu}{\gamma\, \sigma^2} $$

- $\mu$, $\sigma^2$: the premium’s mean and variance. $U$: the sure
  return worth as much. $\gamma$: dislike of risk (3 is typical).

In [ ]:
# Utility of each weight w in stocks, the rest in T-bills (the T-bill part, r_f,
# is left out: it does not move the best w). Risk aversion 3 is a common textbook value.
risk_aversion = 3
expected_premium = annual_premium.mean()        # 1926-2024, per year
variance = annual_premium.std() ** 2            # the sd, squared
best_weight = expected_premium / (risk_aversion * variance)
weights = np.linspace(0, 1.6, 161)              # 0.00, 0.01, ..., 1.60
utility = weights * expected_premium - risk_aversion / 2 * weights ** 2 * variance
best_utility = best_weight * expected_premium - risk_aversion / 2 * best_weight ** 2 * variance
plt.figure(figsize=(9, 2.0))
plt.plot(weights, 100 * utility, color="tab:blue", linewidth=3)
plt.plot(best_weight, 100 * best_utility, "o", color="tab:blue", markersize=10)
plt.text(best_weight + 0.05, 100 * best_utility + 0.3, f"best weight {best_weight:.2f}",
         color="tab:blue")
plt.axhline(0, color="black", linewidth=0.8)
plt.ylim(-2.5, 5.5)
plt.xlabel("weight in stocks, w")
plt.ylabel("U (% a year)")
plt.show()

## Predict: Half the Premium

- The expected premium halves. The risk stays the same.
- **Predict first:** what happens to the best weight?

1.  Halves
2.  Stays the same
3.  Drops to zero
4.  Doubles

In [ ]:
half_premium = expected_premium / 2
half_weight = half_premium / (risk_aversion * variance)
plt.figure(figsize=(9, 2.2))
for premium, color, label in [(expected_premium, "tab:blue", "full premium"),
                              (half_premium, "tab:orange", "half the premium")]:
    best = premium / (risk_aversion * variance)
    plt.plot(weights, 100 * (weights * premium - risk_aversion / 2 * weights ** 2 * variance),
             color=color, linewidth=3)
    peak = best * premium - risk_aversion / 2 * best ** 2 * variance
    plt.plot(best, 100 * peak, "o", color=color, markersize=10)
    plt.text(best + 0.04, 100 * peak + 0.3, f"{label}: {best:.2f}", ha="left", color=color)
plt.axhline(0, color="black", linewidth=0.8)
plt.ylim(-2.5, 5.5)
plt.xlabel("weight in stocks, w")
plt.ylabel("U (% a year)")
plt.show()

## Check: The Best Stock Weight

- Expected premium $\mu$: 6% a year.
- Standard deviation $\sigma$: 20%.
- Risk aversion $\gamma$: 3.
- **Fingers up:** the best weight in stocks,
  $w^* = \mu / (\gamma \sigma^2)$, is:
  1.  0.10
  2.  0.50
  3.  1.50
  4.  0.17

# What a Price Ratio Must Predict

## What Moves Stock Prices So Much?

- To forecast returns, start with prices: the S&P 500’s price moves by
  roughly a fifth in a typical year.

<!-- -->

- **Fingers up:** what moves prices that much?
  1.  News about future profits and dividends
  2.  A change in the return investors demand
  3.  Bubbles
  4.  No idea

## Gordon, Rearranged: $D/P = r - g$

The Gordon growth model: a stock pays dividend $D$ next year, growing at
rate $g$ forever, and investors expect the return $r$:

$$ P = \frac{D}{r - g} \qquad\Longrightarrow\qquad \frac{D}{P} = r - g $$

- **Why:** a return is the dividend yield plus the price growth. Here
  the price grows with the dividend, at $g$, so $r = D/P + g$.
- A high D/P means a **high expected return** $r$, or **low expected
  growth** $g$. A bubble is outside the formula.
- Week 1’s warm-up: a \$0.50 dividend and a \$3.50 price rise on \$100
  is 0.5% + 3.5% = 4.0%. In the data, `ret` includes dividends; `retx`
  does not.
- On this slide D/P is the plain ratio (3%, 4%). Every regression uses
  **log** D/P, as in week 1.

## Check: D/P Rises From 3% to 4%

- Growth $g$ stays at 5%, and next year’s dividend stays the same.
- D/P rises from 3% to 4%.
- **Fingers up:** by $D/P = r - g$, what happens?
  1.  $r$ rises to 9%; the price falls by a quarter
  2.  $r$ rises to 9%; the price rises by a third
  3.  $r$ falls to 7%; the price falls by a quarter
  4.  $r$ stays at 8%; the price does not change

# Does D/P Predict Next Year’s Return?

## D/P Moves, So $r$ or $g$ Must

- If $r$ and $g$ never changed, D/P would never move. It moves a lot:
  since 1871 it has ranged from about 1% to about 15%.
- **The 1970s view:** the return investors demand is constant, so prices
  move only on news about dividends.
- **Shiller (1981):** prices move five to thirteen times more than
  dividend news can justify.
- So expected returns must move too: the “discount rates vary” of
  Cochrane (2011).

## Campbell–Shiller: D/P Is a Forecast

Gordon, with $r$ and $g$ free to change every year (Campbell & Shiller
1988), in logs:

$$ \log(D/P)_t \;\approx\; c + (r_{t+1} - \Delta d_{t+1}) + \rho\,(r_{t+2} - \Delta d_{t+2}) + \rho^2\,(r_{t+3} - \Delta d_{t+3}) + \cdots $$

- $r$ and $\Delta d$: the return and the dividend growth, in logs, in
  each year after $t$.
- $\rho$ is a number just below 1 (about 0.97 here): later years count a
  little less.

<!-- -->

- A high D/P today must be followed by **high returns** or **low
  dividend growth**.
- It is accounting, not a theory. It holds for any dividend-paying stock
  or index, unless prices ride a bubble that never bursts.
- So D/P **must** forecast one or the other, or both.

## Returns or Dividend Growth: Test Returns First

- Campbell–Shiller: D/P must forecast returns, dividend growth, or both.
  Start with returns.
- Week 1, 1927–2024: log D/P had a *t* statistic below 1.96. Not
  significant.
- **Predict first:** on Cochrane’s post-war years, 1947–2009, will *t*
  clear 1.96?
  1.  Yes
  2.  No

## Cochrane’s Regression, Rebuilt

In [ ]:
# Cochrane (2011), Table I: the post-war years, 1947-2009.
# y: this year's premium (simple, decimals). x: log D/P at the end of LAST year.
sample = annual.loc[1947:2009]
X = sm.add_constant(sample["log_dp_last_year"])
fit = sm.OLS(sample["equity"], X).fit()
slope = fit.params["log_dp_last_year"]
t_stat = fit.tvalues["log_dp_last_year"]
r2 = fit.rsquared
print(f"{len(sample)} years: slope {slope:.3f}, t {t_stat:.2f}, R2 {r2:.3f}")

- Post-war, the printed *t* clears 1.96.
- So D/P forecasts next year’s premium.
- **Slope:** multiply the printed slope by 0.1: that is how much higher
  next year’s premium is, in decimals, when log D/P is 0.1 higher (D/P
  about 10% higher).
- Cochrane (2011), Table I: *t* 2.6, $R^2$ 0.09. Compare with the
  printed *t* and $R^2$.

## Your Turn: Swap the Target

- D/P forecasts returns. Does it also forecast dividend growth?
- **Predict first:** if it did, which sign would the slope have?
  1.  Positive
  2.  Negative

``` python
sample = annual.loc[1947:2009]         # Cochrane's years
X = sm.add_constant(sample["log_dp_last_year"])
y = sample["________"]   # the growth column Setup added
fit = sm.OLS(y, X).fit()
slope = fit.params["log_dp_last_year"]
t_stat = fit.tvalues["log_dp_last_year"]
print(f"slope {slope:.3f}, t {t_stat:.2f}")
```

## The Dog That Didn’t Bark

In [ ]:
sample = annual.loc[1947:2009]         # Cochrane's years
X = sm.add_constant(sample["log_dp_last_year"])
y = sample["div_growth"]               # dividend growth, simple
fit = sm.OLS(y, X).fit()
slope = fit.params["log_dp_last_year"]
t_stat = fit.tvalues["log_dp_last_year"]
print(f"slope {slope:.3f}, t {t_stat:.2f}")

- The slope is **positive**: the wrong sign (forecasting growth needs a
  negative one), and *t* is below 1.96.
- So post-war, D/P’s swings were mostly about **expected returns**: the
  return investors demand.
- Cochrane (2008), *The Dog That Did Not Bark*: Sherlock Holmes’s clue
  was a dog that stayed silent. Here, the silent forecast is dividends.

## So Why Does D/P Move?

How the split is computed. Three one-year regressions on last year’s log
D/P: the log return, log dividend growth, and log D/P itself (its
persistence). Because D/P moves slowly, its pull adds up over all the
years ahead: the long-run slope is the one-year slope divided by
$1 - \rho \times \text{persistence}$ (a geometric series). By
Campbell–Shiller, log D/P equals future returns MINUS future dividend
growth, so the long-run slope on returns and MINUS the long-run slope on
dividend growth are the two shares of D/P’s swings. They add up to about
1, that is 100%: nothing else is left.

In [ ]:
# Campbell-Shiller works in LOGS, so this cell uses log returns and log dividend growth.
logs = pd.DataFrame(index=annual.index)
logs["log_return"] = np.log(1 + annual["ret"])                          # dividends included
logs["log_div_growth"] = np.log(annual["d12"] / annual["d12"].shift(1))
logs["log_dp"] = annual["log_dp"]
logs["log_dp_last_year"] = annual["log_dp_last_year"]
sample = logs.loc[1947:2009]                                            # Cochrane's years
X = sm.add_constant(sample["log_dp_last_year"])
slope_return = sm.OLS(sample["log_return"], X).fit().params["log_dp_last_year"]
slope_growth = sm.OLS(sample["log_div_growth"], X).fit().params["log_dp_last_year"]
persistence = sm.OLS(sample["log_dp"], X).fit().params["log_dp_last_year"]
# rho: how much less a year further ahead counts, 1 / (1 + D/P) at the average log D/P
rho = 1 / (1 + np.exp(sample["log_dp_last_year"].mean()))              # about 0.97
# The pull adds up over the years: 1 + rho*persistence + (rho*persistence)**2 + ...
adds_up = 1 / (1 - rho * persistence)
share_returns = slope_return * adds_up            # share of D/P's swings: future returns
share_growth = -slope_growth * adds_up            # minus: D/P = returns MINUS growth
plt.figure(figsize=(9, 2.3))
plt.barh(["dividend growth", "future returns"], [100 * share_growth, 100 * share_returns],
         color=["tab:orange", "tab:blue"])
plt.text(100 * share_returns + 3, 1, f"{share_returns:.0%}", va="center", fontsize=20)
plt.text(100 * share_growth - 3, 0, f"{share_growth:.0%}", va="center", ha="right", fontsize=20)
plt.axvline(0, color="black", linewidth=0.8)
plt.axvline(100, color="gray", linestyle="--", linewidth=1)
plt.xlim(-60, 160)
plt.xlabel("share of the swings in log D/P (%), 1947–2009; dashed: 100")
plt.show()

- Post-war, essentially **all** of D/P’s swings are about future
  returns: the return investors demand (answer 2 in the vote on “What
  Moves Stock Prices So Much?”).
- Dividend growth even goes slightly the wrong way (the dog’s sign), so
  returns carry more than 100%.
- Cochrane (2011): “What we expected to be zero is one; what we expected
  to be one is zero.” (One = all of the swings.)

## Check: What Campbell–Shiller Says

- **Fingers up:** by the Campbell–Shiller identity, a high D/P today
  must be followed by:
  1.  high returns, or low dividend growth
  2.  low returns, or high dividend growth
  3.  high returns and high dividend growth
  4.  nothing in particular: D/P is noise

# Longer Horizons, and Where the Sample Ends

## Predict: Five Years Instead of One

- Same predictor: log D/P at the end of last year.
- New target: the premium **compounded over the next five years**: what
  stocks grew minus what bills grew.
- **Predict first:** is the $R^2$ bigger, smaller, or about the same?
  1.  Bigger
  2.  Smaller
  3.  About the same

## Over Five Years, $R^2$ Nearly Triples

In [ ]:
def excess_return_over_next(k, first_year):
    """Stocks' growth minus bills' growth over the k years from first_year on."""
    stock_growth = 1.0
    bill_growth = 1.0
    for year in range(first_year, first_year + k):
        stock_growth = stock_growth * (1 + annual.loc[year, "ret"])    # simple returns compound
        bill_growth = bill_growth * (1 + annual.loc[year, "Rfree"])
    return stock_growth - bill_growth

horizon_r2 = {}                     # a dictionary: horizon in years -> R2
for k in [1, 2, 3, 4, 5, 6, 7]:
    targets = []
    predictors = []
    # first years 1947, 1948, ...: the last window, first_year to first_year + k - 1, ends in 2009
    for first_year in range(1947, 2009 - k + 2):
        targets.append(excess_return_over_next(k, first_year))
        predictors.append(annual.loc[first_year, "log_dp_last_year"])   # known before the window
    X = sm.add_constant(np.array(predictors))
    fit = sm.OLS(np.array(targets), X).fit()
    horizon_r2[k] = fit.rsquared

plt.figure(figsize=(9, 2.7))
plt.bar([str(k) for k in horizon_r2], list(horizon_r2.values()), color="tab:blue")
for position, value in enumerate(horizon_r2.values()):
    plt.text(position, value + 0.01, f"{value:.2f}", ha="center", fontsize=19)
plt.ylim(0, 0.5)
plt.xlabel("horizon k (years), windows 1947–2009")
plt.ylabel("R²")
plt.show()

- Why: log D/P is slow (regress it on its own last year: the slope is
  close to 1), so a small pull each year adds up.
- Neighboring five-year windows share four years, so the five-year
  regression has far fewer separate histories than windows.

<!-- -->

- Cochrane (1999): a forecast that Chicago warms a third of a degree a
  day in spring explains little of any one day, but almost all of
  January to July.

## Your Turn: Move the End Year

- The same regression, with 15 more years than Cochrane’s.
- **Predict first:** does *t* stay above 1.96?
  1.  Stays above
  2.  Falls below

``` python
sample = annual.loc[1947:________]   # the last year of annual
X = sm.add_constant(sample["log_dp_last_year"])
fit = sm.OLS(sample["equity"], X).fit()
t_stat = fit.tvalues["log_dp_last_year"]
print(f"t {t_stat:.2f}, R2 {fit.rsquared:.3f}")
```

## Where the Sample Ends

In [ ]:
# The same regression on three samples: Cochrane's, yours, and week 1's.
samples = [("Cochrane", 1947, 2009), ("Your cell", 1947, 2024),
           ("Week 1", 1927, 2024)]
for label, first, last in samples:
    sample = annual.loc[first:last]
    X = sm.add_constant(sample["log_dp_last_year"])
    fit = sm.OLS(sample["equity"], X).fit()
    print(f"{label:<10}{first}-{last}: "
          f"t {fit.tvalues['log_dp_last_year']:.2f}, R2 {fit.rsquared:.3f}")

- Same predictor, same method, three samples.
- The verdict depends on where the sample ends, and on where it starts.

## Would You Sell?

In [ ]:
# The latest D/P: the last month of the monthly file (December 2025).
log_dp_now = np.log(monthly["d12"].iloc[-1]) - np.log(monthly["Index"].iloc[-1])
next_year = monthly.index[-1].year + 1
print(f"D/P at the end of {monthly.index[-1]:%B %Y}: {np.exp(log_dp_now):.2%}")
for first, last in [(1947, 2009), (1947, 2024)]:
    sample = annual.loc[first:last]
    fit = sm.OLS(sample["equity"], sm.add_constant(sample["log_dp_last_year"])).fit()
    forecast = fit.params["const"] + fit.params["log_dp_last_year"] * log_dp_now
    print(f"fitted on {first}-{last}: {next_year} premium {forecast:+.1%}")

- Put yourself at the end of 2025. The same regression, fitted on two
  samples, gives two forecasts for 2026.
- **Fingers up:** the first says stocks lose to T-bills. Your move?
  1.  Sell stocks and hold bills
  2.  Keep holding
  3.  Not sure

## Check: Why $R^2$ Rises With the Horizon

- **Fingers up:** why is the $R^2$ bigger at five years than at one?
  1.  D/P moves slowly, so its small pull on each year adds up
  2.  A five-year premium swings less than a one-year premium
  3.  Neighboring windows share years, which adds information
  4.  There are more five-year observations than one-year ones

# Is a Small $R^2$ Worth Anything?

## An $R^2$ of 0.10: Useless, or Worth Money?

- Post-war, log D/P explains about a tenth of the **variation** in next
  year’s premium (the $R^2$ of “Cochrane’s Regression, Rebuilt”).
- **Fingers up:** week 1 called an $R^2$ of about 0.02 tiny. Is 0.10
  useless, or worth money?
  1.  Useless
  2.  Worth money
  3.  Not sure

## How Much Does the Forecast Move?

- The spread of the forecast = slope × the spread of log D/P.
- Put a low and a high forecast into the best-weight rule: risk aversion
  3, the variance of 1926–2024.

In [ ]:
post_war = annual.loc[1947:2009]
X = sm.add_constant(post_war["log_dp_last_year"])
fit = sm.OLS(post_war["equity"], X).fit()
forecast = fit.fittedvalues                 # each year's forecast of the premium
slope = fit.params["log_dp_last_year"]
spread_of_log_dp = post_war["log_dp_last_year"].std()
print(f"forecast: mean {forecast.mean():.2%}, sd {forecast.std():.2%}")
print(f"sd = slope {slope:.3f} x sd of log D/P {spread_of_log_dp:.3f}")
risk_aversion = 3
variance = annual.loc[1926:2024, "equity"].std() ** 2   # as on "Forecasts Set the Stock Weight"
low_weight = (forecast.mean() - forecast.std()) / (risk_aversion * variance)
high_weight = (forecast.mean() + forecast.std()) / (risk_aversion * variance)
print(f"stock weight at mean - 1 sd: {low_weight:.2f}")
print(f"stock weight at mean + 1 sd: {high_weight:.2f}")

- An $R^2$ of 0.10 moves the best weight a lot.
- Above 1, you **borrow** to hold more than 100% in stocks.

## From $R^2$ to Sharpe Ratio

- **Sharpe ratio** $S$ = average premium / its sd: the reward per unit
  of risk.
- Annual, 1926–2024: the mean and sd of “The Premium Is Big, and Noisy”
  give $S$ a little above 0.4.
- A market timer whose forecast has this $R^2$ can reach, at most, about

$$ S^* = \sqrt{\frac{S^2 + R^2}{1 - R^2}} $$

- $R^2 = 0$ gives $S^* = S$: no forecast, no gain (Cochrane 1999,
  *Portfolio Advice for a Multifactor World*; Campbell & Thompson 2008).

<!-- -->

- Post-war, 1947–2009: the $R^2$ of “Cochrane’s Regression, Rebuilt”
  would lift $S$ by about a quarter. Try it: put your own numbers into
  the formula.

## Hindsight Against Real Time

The certainty equivalent is $U$ from “Forecasts Set the Stock Weight”,
average − (risk aversion / 2) × variance: the sure return over T-bills
worth as much to this investor. Two investors each year from 1960 to
2024: one ignores D/P (the historical mean sets the weight), one times
the market with the D/P forecast. With hindsight, the slope is fitted on
1960–2024 itself; in real time, each year’s slope uses only the years
before it (week 1’s loop).

In [ ]:
def certainty_equivalent(portfolio_premiums, risk_aversion=3):
    """Average minus (risk aversion / 2) x variance: the sure return worth the same."""
    return np.mean(portfolio_premiums) - risk_aversion / 2 * np.var(portfolio_premiums, ddof=1)

data = pd.DataFrame({"premium": annual["equity"],
                     "predictor": annual["log_dp_last_year"]}).loc[1927:2024]
years = data.loc[1960:2024]                                   # the years both investors trade

# 1. With hindsight: one regression fitted on 1960-2024 itself sets every weight.
hindsight_fit = sm.OLS(years["premium"], sm.add_constant(years["predictor"])).fit()
hindsight_variance = years["premium"].var()
timing = hindsight_fit.fittedvalues / (3 * hindsight_variance) * years["premium"]   # weight x premium
no_timing = years["premium"].mean() / (3 * hindsight_variance) * years["premium"]
hindsight = [certainty_equivalent(no_timing), certainty_equivalent(timing)]

# 2. In real time: week 1's loop from 1960; each year uses only the years before it.
start_row = data.index.get_loc(1960)
timing_real = []
no_timing_real = []
for t in range(start_row, len(data)):
    training = data.iloc[:t]                              # years BEFORE row t: no peeking
    real_fit = sm.OLS(training["premium"], sm.add_constant(training["predictor"])).fit()
    forecast_now = real_fit.params["const"] + real_fit.params["predictor"] * data["predictor"].iloc[t]
    variance_now = training["premium"].var()
    premium_now = data["premium"].iloc[t]
    timing_real.append(forecast_now / (3 * variance_now) * premium_now)
    no_timing_real.append(training["premium"].mean() / (3 * variance_now) * premium_now)
real_time = [certainty_equivalent(no_timing_real), certainty_equivalent(timing_real)]

positions = [0, 1, 2.6, 3.6]
values = [100 * hindsight[0], 100 * hindsight[1], 100 * real_time[0], 100 * real_time[1]]
plt.figure(figsize=(9, 2.7))
plt.bar(positions, values, width=0.8, color=["0.7", "tab:blue", "0.7", "tab:blue"])
for position, value in zip(positions, values):
    plt.text(position, value + 0.1, f"{value:.1f}%", ha="center", fontsize=20)
plt.xticks(positions, ["ignore D/P", "use D/P", "ignore D/P", "use D/P"])
plt.text(0.5, 4.5, "with hindsight", ha="center", fontsize=20)
plt.text(3.1, 4.5, "in real time", ha="center", fontsize=20)
plt.ylim(0, 5.4)
plt.ylabel("certainty equiv.\n(% a year)")
plt.show()

- With hindsight, timing on D/P helps; in real time, on the same years,
  it hurts.

## On Paper: A Monthly $R^2$ of 2%

- Monthly, the premium’s Sharpe ratio is about 0.13 (week 1’s monthly
  mean over its sd).

<!-- -->

- A monthly forecast has $R^2$ = 2% **out of sample**.
- By how much could the Sharpe ratio rise?

## Check: Hindsight or Real Time?

- **Fingers up:** which is true?
  1.  A regression judged with hindsight looks better than it would have
      in real time
  2.  A positive in-sample $R^2$ guarantees gains in real time
  3.  A real-time forecast uses data from after the forecast
  4.  Hindsight and real time always give the same answer

## In Sample Is Not Enough

- Every regression before “Hindsight Against Real Time” was fitted on
  **all** of its years, the ones it “forecast” included.
- Week 1: the same predictor, fitted only on the past, lost to the
  historical mean from 1960, and trading on it lost in utility terms.

<!-- -->

- **A small $R^2$ pays only if it holds out of sample.** Next: Goyal &
  Welch (2008) put every predictor to that test.

## Before Meeting 4

1.  Read Goyal & Welch (2008). It is required for class.
2.  Each new Colab session: upload `gw_monthly.csv` and `gw_annual.csv`
    again, and run Setup.

## Your turn

On a card, before you leave. No names needed.

1.  Post-war, D/P barely forecasts dividend growth. So what must its
    swings be telling you about, and why?
2.  The post-war regression has an $R^2$ of about 0.10. Give one reason
    it could be worth money, and one thing that must be true first.

# Can Anything Beat the Historical Mean?

## Upload, Then Run Setup

Starting a new Colab session? Colab deletes uploaded files when a
session ends. Download `gw_monthly.csv` and `gw_annual.csv` from
Brightspace, drag them into the Files pane (folder icon on the left),
then run the **Setup** cell at the top of this notebook. Then continue
here: nothing from Meeting 3 needs to run again. If your session
restarts later, run Setup, then every code cell from “Week 1’s Loop, in
a Function” down to where you are.

# Where to Start, and Which Past to Use

## Ten Percent in Sample, Negative in Real Time

- **In sample:** post-war log D/P, 1947–2009, forecast next year’s
  premium with an $R^2$ of about 0.10.
- **Week 1, out of sample:** from 1960 on, $R^2_{\text{OOS}}$ was
  negative.

<!-- -->

- Three choices set that score,
  $R^2_{\text{OOS}} = 1 - \text{MSE}_{\text{model}} / \text{MSE}_{\text{mean}}$:
  - the **window**: expanding, every year before the one forecast;
  - the **benchmark**: the historical mean over those same years;
  - the **start date**: 1960.
- **Questions:** was 1960 just a bad start? Which past should the model
  learn from? Is D/P special, or do most predictors fail?

## Week 1’s Loop, in a Function

Week 1’s loop and score, wrapped in `def ... return` (as `annualize`
was), so one line runs it for any start year. Two answers come back: the
score and the number of forecasts.

In [ ]:
def oos_r2_from(first_year):
    """Forecast each year from first_year to 2024 with log D/P, fitting on
    the years before it only. Returns R2_OOS and the number of forecasts."""
    # Row t: the premium of year t, and log D/P at the end of year t - 1.
    data = pd.DataFrame({"premium": annual["equity"],
                         "predictor": annual["log_dp_last_year"]})
    data = data.loc[1927:2024]
    first_row = data.index.get_loc(first_year)   # the row of the first forecast
    model_forecasts = []        # the regression's forecasts
    mean_forecasts = []         # the historical mean's forecasts
    actual_premiums = []        # what actually happened
    for t in range(first_row, len(data)):
        training = data.iloc[:t]            # years BEFORE row t: no peeking
        X_train = sm.add_constant(training["predictor"])
        fit = sm.OLS(training["premium"], X_train).fit()
        predictor_now = data["predictor"].iloc[t]    # known a year earlier
        forecast = fit.params["const"] + fit.params["predictor"] * predictor_now
        model_forecasts.append(forecast)
        mean_forecasts.append(training["premium"].mean())
        actual_premiums.append(data["premium"].iloc[t])
    actual = np.array(actual_premiums)
    mse_model = np.mean((actual - np.array(model_forecasts)) ** 2)
    mse_mean = np.mean((actual - np.array(mean_forecasts)) ** 2)
    r2_oos = 1 - mse_model / mse_mean
    return r2_oos, len(actual)

score_1960, count_1960 = oos_r2_from(1960)
score_1970, count_1970 = oos_r2_from(1970)
print(f"from 1960: R2_OOS {score_1960:.2%}, {count_1960} forecasts")
print(f"from 1970: R2_OOS {score_1970:.2%}, {count_1970} forecasts")
# A check: we must get week 1's numbers back. assert stops with an error if not.
assert f"{score_1960:.2%}" == "-2.20%" and count_1960 == 65
assert f"{score_1970:.2%}" == "-5.47%" and count_1970 == 55

## Your Turn: Your Assigned Start Year

- Your start year comes from your birth month.
- **Predict first:** which sign will your $R^2_{\text{OOS}}$ have?
  1.  Positive
  2.  Negative

| born in     | Jan     | Feb     | Mar     | Apr     | May     | Jun     |
|:------------|:--------|:--------|:--------|:--------|:--------|:--------|
| start year  | 1948    | 1952    | 1956    | 1960    | 1964    | 1968    |
| **born in** | **Jul** | **Aug** | **Sep** | **Oct** | **Nov** | **Dec** |
| start year  | 1972    | 1976    | 1980    | 1984    | 1988    | 1992    |

``` python
score, count = oos_r2_from(____)   # your start year
print(f"R2_OOS {score:.2%}, {count} forecasts")
```

## The Start Date Moves the Score

In [ ]:
start_years = list(range(1947, 2001))    # every first forecast year, 1947 to 2000
start_scores = []
for year in start_years:
    score, count = oos_r2_from(year)
    start_scores.append(100 * score)     # in percent
above_zero = (np.array(start_scores) > 0).sum()   # True counts as 1
plt.figure(figsize=(10, 5))
plt.plot(start_years, start_scores, color="tab:blue", marker="o")
plt.axhline(0, color="black", linewidth=0.8)      # zero: as good as the historical mean
plt.title(f"Above zero for {above_zero} of {len(start_years)} start years")
plt.xlabel("first forecast year (forecasts run to 2024)")
plt.ylabel("R²_OOS (%)")
plt.show()

- The start year changes the **size** of the loss, and almost never its
  **sign**.

## Expanding or Rolling Window?

- **Expanding:** steadier, but slow to notice a change.
- **Rolling:** adapts faster, but noisier (fewer observations).
- “Where the Sample Ends”: the answer moved with the years used.
- If the relation drifts, old years can mislead.

## Check: Expanding or Rolling?

- **Fingers up:** which is true?
  1.  A rolling window uses more data than an expanding one
  2.  An expanding window adapts faster to a change
  3.  A rolling window adapts faster, but its forecasts are noisier
  4.  The two always give the same forecast

# The Goyal–Welch Verdict, on Our Data

## How Many of Twelve Predictors Beat the Historical Mean?

- Twelve classic predictors, eleven of them from Goyal & Welch (2008),
  **monthly**, each **alone**, in week 1’s loop.

<!-- -->

- Expanding window from February 1926. First forecast January 1970, last
  December 2025: one forecast a month.

<!-- -->

- **Predict first:** how many beat the mean?
  1.  0–2
  2.  3–6
  3.  7–12

## Twelve Predictors, One at a Time

- One cell builds all twelve predictors and runs week 1’s loop once per
  predictor: 12 × 672 regressions.
- Run it now (5 to 10 seconds). It keeps every forecast and every
  month’s estimated slope.
- The result, `forecasts`: one row per month, one column of forecasts
  per predictor. Its first rows:

Run this cell before the ones below: it builds the twelve predictors and
the forecasts that the rest of this meeting’s cells use.

In [ ]:
# Twelve classic predictors (eleven from Goyal and Welch 2008, plus rrel), each as it
# stood at the END of its own month. d12, e12: S&P 500 dividends and earnings over the past 12 months;
# Index: the S&P 500 price at the end of the month.
unlagged = pd.DataFrame(index=monthly.index)
# dp, dividend-price ratio: log(dividends) - log(price)
unlagged["dp"] = np.log(monthly["d12"]) - np.log(monthly["Index"])
# dy, dividend yield: log(dividends) - log(LAST month's price)
unlagged["dy"] = np.log(monthly["d12"]) - np.log(monthly["Index"].shift(1))
# ep, earnings-price ratio: log(earnings) - log(price)
unlagged["ep"] = np.log(monthly["e12"]) - np.log(monthly["Index"])
# svar, stock variance: sum of squared daily S&P 500 returns in the month
unlagged["svar"] = monthly["svar"]
# bm, book-to-market ratio of the Dow Jones Industrial Average (a ratio, not a log)
unlagged["bm"] = monthly["b/m"]
# tbl, 3-month Treasury-bill yield (annual rate, decimal)
unlagged["tbl"] = monthly["tbl"]
# lty, long-term government bond yield (annual rate, decimal)
unlagged["lty"] = monthly["lty"]
# ltr, long-term government bond return (monthly, decimal)
unlagged["ltr"] = monthly["ltr"]
# dfy, default yield spread: BAA minus AAA corporate bond yield
unlagged["dfy"] = monthly["BAA"] - monthly["AAA"]
# dfr, default return spread: corporate minus government long-term bond return
unlagged["dfr"] = monthly["corpr"] - monthly["ltr"]
# infl, CPI inflation: a month's CPI comes out during the NEXT month, so one extra lag
unlagged["infl"] = monthly["infl"].shift(1)
# rrel, relative T-bill rate: tbl minus its average over the last 3 months (this one included)
unlagged["rrel"] = monthly["tbl"] - monthly["tbl"].rolling(window=3).mean()
predictor_names = list(unlagged.columns)     # the twelve names, in order

# Lag every predictor one month: a forecast for month t uses only the end of t - 1.
table = unlagged.shift(1)
# The target: the month's premium, a SIMPLE excess return in decimals, never annualized.
table["premium"] = monthly["equity"]
table = table.loc["1926-02":"2025-12"]       # the premium starts 1926-01; the lag costs one month

# Week 1's loop, month by month: expanding window, first forecast January 1970.
first_row = table.index.get_loc(pd.Timestamp("1970-01-01"))
forecasts = pd.DataFrame(index=table.index[first_row:])   # one row per forecast month
forecasts.index.name = "month"
forecasts["actual"] = table["premium"].iloc[first_row:]   # what actually happened
mean_forecasts = []
for t in range(first_row, len(table)):
    mean_forecasts.append(table["premium"].iloc[:t].mean())   # the historical mean
forecasts["historical_mean"] = mean_forecasts
slopes = pd.DataFrame(index=forecasts.index)  # each month's estimated slope, per predictor
for name in predictor_names:                 # one regression per predictor, alone
    model_forecasts = []
    model_slopes = []
    for t in range(first_row, len(table)):
        training = table.iloc[:t]            # months BEFORE row t: no peeking
        X_train = sm.add_constant(training[name])
        fit = sm.OLS(training["premium"], X_train).fit()
        predictor_now = table[name].iloc[t]  # known at the end of last month
        forecast = fit.params["const"] + fit.params[name] * predictor_now
        model_forecasts.append(forecast)
        model_slopes.append(fit.params[name])
    forecasts[name] = model_forecasts        # one column per predictor
    slopes[name] = model_slopes
print(forecasts[["actual", "historical_mean", "dp", "tbl"]].head(2).round(4))

## Almost Nothing Beats the Mean

Each bar: one predictor alone, scored with week 1’s $R^2_{\text{OOS}}$
against the historical mean. Left of zero: it lost.

In [ ]:
labels = {"dp": "dividend-price ratio", "dy": "dividend yield",
          "ep": "earnings-price ratio", "svar": "stock variance",
          "bm": "book-to-market", "tbl": "T-bill rate",
          "lty": "long-term bond yield", "ltr": "long-term bond return",
          "dfy": "default yield spread", "dfr": "default return spread",
          "infl": "inflation", "rrel": "relative T-bill rate"}
mse_mean = np.mean((forecasts["actual"] - forecasts["historical_mean"]) ** 2)
scores = {}                                  # a dictionary: label -> R2_OOS in percent
for name in predictor_names:
    mse_model = np.mean((forecasts["actual"] - forecasts[name]) ** 2)
    scores[labels[name]] = 100 * (1 - mse_model / mse_mean)
scores = pd.Series(scores).sort_values()     # smallest first: it lands at the bottom
below_zero = (scores < 0).sum()              # True counts as 1

plt.figure(figsize=(10, 5.2))
plt.barh(scores.index, scores, color="tab:blue")
plt.axvline(0, color="black", linewidth=0.8)   # zero: as good as the historical mean
for row in range(len(scores)):                 # write each value next to its bar
    value = scores.iloc[row]
    if value < 0:
        plt.text(value - 0.05, row, f"{value:.2f}", va="center", ha="right", fontsize=16)
    else:
        plt.text(value + 0.05, row, f"+{value:.2f}", va="center", ha="left", fontsize=16)
plt.xlim(scores.min() - 1.0, scores.max() + 0.7)
plt.title(f"{below_zero} of {len(scores)} lose to the historical mean")
plt.xlabel(f"R²_OOS (%), {len(forecasts)} monthly forecasts, "
           f"{forecasts.index[0].year}–{forecasts.index[-1].year}")
plt.show()

## Check: What a Negative $R^2_{\text{OOS}}$ Means

- The dividend-price ratio scores $R^2_{\text{OOS}}$ = −0.9%.
- **Fingers up:** that means:
  1.  its squared forecast errors were 0.9% larger than the historical
      mean’s
  2.  it explained −0.9% of returns in sample
  3.  the premium fell by 0.9% a month
  4.  it beat the historical mean by 0.9%

# Use What You Know

## Never Forecast a Negative Premium

- Someone must hold every share. If investors expected stocks to lose to
  T-bills, nobody would, so the expected premium is never below zero
  (Campbell & Thompson 2008).
- So floor every forecast at zero: `forecasts["dp"].clip(lower=0)`. On
  “Would You Sell?”, that means hold no stocks, not short them.
- **Predict first:** does the floor help or hurt $R^2_{\text{OOS}}$?
  1.  Helps
  2.  Hurts

In [ ]:
floored = forecasts["dp"].clip(lower=0)      # a forecast below 0 becomes 0
mse_mean = np.mean((forecasts["actual"] - forecasts["historical_mean"]) ** 2)
mse_dp = np.mean((forecasts["actual"] - forecasts["dp"]) ** 2)
mse_floored = np.mean((forecasts["actual"] - floored) ** 2)
print(f"R2_OOS, dividend-price ratio: {1 - mse_dp / mse_mean:.2%}")
print(f"R2_OOS, floored at zero:      {1 - mse_floored / mse_mean:.2%}")
print(f"months the floor changed: {(forecasts['dp'] < 0).sum()} of {len(forecasts)}")

## Theory Knows the Sign

- Theory gives each slope a sign: a high D/P means **higher** returns, a
  high T-bill rate **lower**.
- Two rules: a wrong-sign month uses the historical mean, plus the zero
  floor.

In [ ]:
# The sign each slope should have: +1 means "a higher value, a higher premium".
theory_sign = {
    "dp": +1, "dy": +1, "ep": +1, "bm": +1,   # cheap prices (Gordon): higher returns ahead
    "svar": +1,                               # more risk, more reward
    "tbl": -1, "lty": -1, "rrel": -1,         # high or rising rates: lower stock returns
    "ltr": +1,        # a high bond return means yields fell: lower rates, higher returns ahead
    "dfy": +1,        # a wide credit spread: bad times, a higher premium
    "dfr": +1,        # no clear theory: the sign the data show
    "infl": -1,       # inflation hurts stocks (Fama and Schwert 1977)
}
raw_scores = {}
restricted_scores = {}
for name in predictor_names:
    wrong_sign = np.sign(slopes[name]) != theory_sign[name]   # True in a month with the wrong sign
    # .where(condition, other): keep the forecast where the condition holds (~ means "not"),
    # else take the historical mean.
    restricted = forecasts[name].where(~wrong_sign, forecasts["historical_mean"])
    restricted = restricted.clip(lower=0)                     # then the zero floor
    raw_scores[labels[name]] = 100 * (1 - np.mean((forecasts["actual"] - forecasts[name]) ** 2) / mse_mean)
    restricted_scores[labels[name]] = 100 * (1 - np.mean((forecasts["actual"] - restricted) ** 2) / mse_mean)
raw_scores = pd.Series(raw_scores).sort_values()
restricted_scores = pd.Series(restricted_scores)[raw_scores.index]   # same order
rows = np.arange(len(raw_scores))
plt.figure(figsize=(10, 3.4))
plt.barh(rows + 0.2, raw_scores, height=0.4, color="0.7")
plt.barh(rows - 0.2, restricted_scores, height=0.4, color="tab:blue")
plt.yticks(rows, raw_scores.index, fontsize=16)
plt.axvline(0, color="black", linewidth=0.8)
plt.title(f"gray: as estimated ({(raw_scores < 0).sum()} lose)   "
          f"blue: both rules ({(restricted_scores < 0).sum()} lose)", fontsize=17)
plt.xlabel("R²_OOS (%), monthly, 1970–2025")
plt.show()

- Finance cuts the losses but makes no winners.

## Gordon, as a Forecast

- Gordon, turned around: expected return = D/P + $g$. Subtract the
  T-bill yield, and it forecasts the premium.
- Forecast for next month = (D/P + average dividend growth so far −
  T-bill yield) / 12. **No slope is estimated from returns.**

In [ ]:
# Everything is known at the end of LAST month: D/P, the T-bill yield, and the
# average 12-month dividend growth over all months so far. All are per year.
dp_level = (monthly["d12"] / monthly["Index"]).shift(1)            # D/P, not in logs
growth_12_months = monthly["d12"] / monthly["d12"].shift(12) - 1   # this year's dividends / last year's
average_growth = growth_12_months.expanding().mean().shift(1)      # the average of all months so far
tbl_last_month = monthly["tbl"].shift(1)
gordon = ((dp_level + average_growth - tbl_last_month) / 12).loc[forecasts.index]   # per month
mse_gordon = np.mean((forecasts["actual"] - gordon) ** 2)
print(f"R2_OOS, Gordon, nothing estimated: {1 - mse_gordon / mse_mean:.2%}")
print(f"R2_OOS, dividend-price regression: {1 - mse_dp / mse_mean:.2%}")

- Still below zero: a much smaller loss, not a win.
- Campbell & Thompson (2008): fixing coefficients with theory can beat
  estimating them.

## Why Restrictions Help

- A slope estimated from noisy returns is itself noisy. Now and then it
  produces a wild forecast.
- Restrictions cut off the wildest forecasts and pull the rest toward
  something sensible.
- That is **shrinkage by economics**. Ridge and lasso (Meetings 5 and 6)
  do it by statistics.

## Check: What the Zero Floor Does

- **Fingers up:** why can flooring forecasts at zero raise
  $R^2_{\text{OOS}}$?
  1.  It moves forecasts that were on the wrong side of zero toward what
      usually happens
  2.  It uses data from after the forecast
  3.  It raises the in-sample $R^2$
  4.  Stocks never lose to T-bills

# More Predictors, Worse Forecasts

## If Each Predictor Knows a Little, Why Not Use Them All?

- Week 1 (“In-Sample $R^2$ Is Not the Criterion”): pure noise raised
  in-sample $R^2$ from 26.3% to 34.5%.

<!-- -->

- Now all twelve predictors in **one** regression, on **annual** data,
  forecasting from 1960 like week 1’s investor. Log D/P alone scored
  about −2%.

<!-- -->

- **Predict first:** how do all twelve together score?
  1.  Better
  2.  About the same
  3.  Worse
  4.  Much worse

## All Twelve at Once

The same twelve predictors, once a year, each as it stood at the end of
the year: the ratios from each December, yields from December, and the
year’s totals for returns, inflation and variance.

In [ ]:
# The same twelve predictors, once a year: each as it stood at the END of the year.
december = monthly[monthly.index.month == 12].copy()
december.index = december.index.year                 # label the rows by year, like annual
yearly = pd.DataFrame(index=annual.index)
yearly["dp"] = np.log(december["d12"]) - np.log(december["Index"])
yearly["dy"] = np.log(december["d12"]) - np.log(december["Index"].shift(1))   # last December's price
yearly["ep"] = np.log(december["e12"]) - np.log(december["Index"])
yearly["svar"] = annual["svar"]                      # the year's sum of squared daily returns
yearly["bm"] = december["b/m"]
yearly["tbl"] = december["tbl"]
yearly["lty"] = december["lty"]
yearly["ltr"] = annual["ltr"]                        # the year's long-term bond return
yearly["dfy"] = december["BAA"] - december["AAA"]
yearly["dfr"] = annual["corpr"] - annual["ltr"]      # the year's corporate minus government return
yearly["infl"] = annual["infl"]                      # the year's inflation (published in January)
rrel_december = unlagged.loc[unlagged.index.month == 12, "rrel"]   # the monthly rrel, each December
rrel_december.index = rrel_december.index.year
yearly["rrel"] = rrel_december                       # pandas lines the rows up by year
yearly_table = yearly.shift(1)                       # a forecast for year t uses the end of t - 1
yearly_table["premium"] = annual["equity"]           # SIMPLE excess return of year t, decimals
yearly_table = yearly_table.loc[1927:2024]           # 1927: the first year with all twelve

# Week 1's loop, with twelve predictors at once, first forecast 1960.
start_row = yearly_table.index.get_loc(1960)
all_twelve = []
yearly_mean = []
yearly_actual = []
for t in range(start_row, len(yearly_table)):
    training = yearly_table.iloc[:t]                 # years BEFORE row t: no peeking
    fit = sm.OLS(training["premium"], sm.add_constant(training[predictor_names])).fit()
    x_now = yearly_table[predictor_names].iloc[t]
    all_twelve.append(fit.params["const"] + (fit.params[predictor_names] * x_now).sum())
    yearly_mean.append(training["premium"].mean())
    yearly_actual.append(yearly_table["premium"].iloc[t])
yearly_actual = np.array(yearly_actual)
mse_yearly_mean = np.mean((yearly_actual - np.array(yearly_mean)) ** 2)
r2_all_twelve = 1 - np.mean((yearly_actual - np.array(all_twelve)) ** 2) / mse_yearly_mean
plt.figure(figsize=(9, 2.6))
plt.barh(["all twelve at once", "log D/P alone"], [100 * r2_all_twelve, 100 * score_1960],
         color=["tab:orange", "tab:blue"])
plt.text(100 * r2_all_twelve + 2, 0, f"{r2_all_twelve:.0%}", va="center", fontsize=20)
plt.text(100 * score_1960 - 2, 1, f"{score_1960:.1%}", va="center", ha="right", fontsize=20)
plt.axvline(0, color="black", linewidth=0.8)
plt.xlim(-130, 15)
plt.xlabel("R²_OOS (%), annual forecasts, 1960–2024")
plt.show()

- All twelve at once lose far more than log D/P alone.
- Their squared forecast errors are more than twice the historical
  mean’s.

## Two Suspects

- **Suspect 1:** the twelve predictors are useless.
- **Suspect 2:** twelve slopes from 33 to 97 years of data are too many
  to estimate.
- Test suspect 2 alone: swap the twelve for twelve columns of **pure
  noise**, same years, same loop.

In [ ]:
# Twelve columns of random numbers, one number per year. The fixed seed gives everyone the same noise.
generator = np.random.default_rng(2026)
yearly_noise = pd.DataFrame(generator.normal(size=(len(yearly_table), 12)),
                            index=yearly_table.index)
X_noise = sm.add_constant(yearly_noise)              # an intercept plus the twelve noise columns
noise_forecasts = []
for t in range(start_row, len(yearly_table)):
    fit = sm.OLS(yearly_table["premium"].iloc[:t], X_noise.iloc[:t]).fit()   # years BEFORE t
    noise_forecasts.append((fit.params * X_noise.iloc[t]).sum())   # each coefficient x its value
r2_noise_yearly = 1 - np.mean((yearly_actual - np.array(noise_forecasts)) ** 2) / mse_yearly_mean
print(f"12 noise columns: R2_OOS {r2_noise_yearly:.0%}")
print(f"the real twelve:  R2_OOS {r2_all_twelve:.0%}")

- Noise alone loses a lot: twelve slopes from a few decades cost dearly,
  even with nothing real in them.
- The real twelve lose even more.
- The part that noise does not explain opens Meeting 5.

## Pure Noise, Out of Sample

- **Rule of thumb:** each useless coefficient lowers $R^2_{\text{OOS}}$
  by roughly $1/T$, and more when the columns are many.
- $T$ is the number of training observations: months here, years with
  annual data.
- Why: each slope is estimated from $T$ observations. Its error raises
  the forecast’s mean squared error by about 1 part in $T$.
- Back to monthly, week 1’s loop from 1970: the historical mean plus 1,
  12 or 50 columns of pure noise.

In [ ]:
# 100 columns of random numbers, one number per month. The fixed seed gives
# everyone the same noise.
generator = np.random.default_rng(2026)
noise_columns = {}                           # a dictionary: name -> column
for number in range(1, 101):
    noise_columns[f"noise_{number}"] = generator.normal(size=len(table))
noise = pd.DataFrame(noise_columns, index=table.index)

def noise_scores(how_many):
    """In-sample R2 and R2_OOS of the premium on an intercept plus the first
    how_many noise columns."""
    # has_constant="add": always add the intercept, even with 0 noise columns
    X = sm.add_constant(noise.iloc[:, :how_many], has_constant="add")
    # In sample: one regression on all the months at once.
    in_sample = sm.OLS(table["premium"], X).fit().rsquared
    # Out of sample: week 1's loop, month by month from 1970.
    model_forecasts = []
    for t in range(first_row, len(table)):
        fit = sm.OLS(table["premium"].iloc[:t], X.iloc[:t]).fit()   # months BEFORE t
        forecast = (fit.params * X.iloc[t]).sum()   # each coefficient x its value, added up
        model_forecasts.append(forecast)
    mse_model = np.mean((forecasts["actual"] - np.array(model_forecasts)) ** 2)
    mse_mean = np.mean((forecasts["actual"] - forecasts["historical_mean"]) ** 2)
    return in_sample, 1 - mse_model / mse_mean

for how_many in [1, 12, 50]:
    in_sample, out_of_sample = noise_scores(how_many)
    label = "column: " if how_many == 1 else "columns:"
    print(f"{how_many:2d} {label} in sample {in_sample:.2%}, R2_OOS {out_of_sample:.2%}")
print(f"T: {first_row} months at the first forecast, {len(table) - 1} at the last")
print(f"1/T: from {1 / first_row:.2%} to {1 / (len(table) - 1):.2%} per useless column")

## Your Turn: Add Your Own Noise

- With no noise at all, both scores are 0. Now add noise.
- **Predict first:** which way does in-sample $R^2$ go? And
  $R^2_{\text{OOS}}$?
  1.  Up
  2.  Down

``` python
how_many = ____     # any whole number from 1 to 100
in_sample, out_of_sample = noise_scores(how_many)
print(f"in-sample R2 {in_sample:.2%}")
print(f"R2_OOS {out_of_sample:.2%}")
```

## In Sample Up, Out of Sample Down

In [ ]:
counts = [0, 1, 2, 5, 12, 25, 50]
in_sample_r2 = []
out_of_sample_r2 = []
for how_many in counts:
    in_sample, out_of_sample = noise_scores(how_many)
    in_sample_r2.append(100 * in_sample)          # in percent
    out_of_sample_r2.append(100 * out_of_sample)
plt.figure(figsize=(10, 5))
plt.plot(counts, in_sample_r2, color="tab:blue", marker="o", linewidth=3)
plt.plot(counts, out_of_sample_r2, color="tab:orange", marker="o", linewidth=3)
plt.axhline(0, color="black", linewidth=0.8)
plt.text(51.5, in_sample_r2[-1], "in sample", color="tab:blue", va="center")
plt.text(51.5, out_of_sample_r2[-1], "out of sample", color="tab:orange", va="center")
plt.xlim(-1, 68)
plt.xlabel("number of pure-noise predictors")
plt.ylabel("R² (%)")
plt.show()

- The rising line is week 1’s slide again. The falling line is new.

## Check: 50 Years or 500 Months?

- Five useless predictors.
- **Fingers up:** roughly how much $R^2_{\text{OOS}}$ do they cost?
  1.  About 10 points with 50 years, about 1 with 500 months
  2.  About 1 point either way
  3.  About 1 point with 50 years, about 10 with 500 months
  4.  Nothing: useless predictors get slopes of zero

# Average the Forecasts

## The Power of Averaging

- **Galton (1907):** 787 fair-goers guessed an ox’s weight. The middle
  guess missed by under 1% (1,207 lb against 1,198).
- **Portfolios:** many stocks cancel much of each one’s own risk.
- **Forecasts:** combining forecasts (Bates & Granger 1969) is one of
  the most reliable gains in forecasting.
- **Machine learning:** a random forest averages the forecasts of
  hundreds of trees.

## Twelve Analysts, One Indicator Each

- Picture twelve analysts, each watching **one** indicator: the
  valuation analyst (D/P), the bond analyst (yields), the macro analyst
  (inflation), and so on.

<!-- -->

- Out of sample, most of the twelve lost to the historical mean (the bar
  chart).

<!-- -->

- Each month, average their twelve forecasts.
- **Predict first:** how does the averaged forecast compare with the
  analysts’ average score?
  1.  Better
  2.  About the same
  3.  Worse

## Two Analysts, by Hand

- **On paper first.** Analyst A forecasts 4%, analyst B 10%. The premium
  turns out 6%.
- What is each analyst’s squared error, and their average?
- The averaged forecast is 7%. What is its squared error?

In [ ]:
actual_premium = 6                           # percent
error_a = actual_premium - 4                 # analyst A forecast 4%
error_b = actual_premium - 10                # analyst B forecast 10%
error_average = actual_premium - (4 + 10) / 2    # the averaged forecast, 7%
print(f"squared errors, A and B: {error_a ** 2}, {error_b ** 2}")
print(f"their average: {(error_a ** 2 + error_b ** 2) / 2:.0f}")
print(f"squared error of the 7% forecast: {error_average ** 2:.0f}")

- The errors partly cancel, like idiosyncratic risk in a portfolio.
- **Always true:** the averaged forecast’s squared error (here 1) is
  never above the analysts’ average squared error (here 10).
- The gain is largest when the analysts disagree.
- Our combination: the plain average of the twelve forecasts.

## Your Turn: Average the Twelve

- **Predict first:** which sign will the average’s $R^2_{\text{OOS}}$
  have?
  1.  Positive
  2.  Negative
- `forecasts[predictor_names]`: one row per month, one column of
  forecasts per analyst (the table on “Twelve Predictors, One at a
  Time”).
- Fill the blank with the pandas method that averages. `axis=1` averages
  **across** the columns, within each row (month).
- **+ Code**, paste, fill the blank, Shift + Enter. Read the printed
  $R^2_{\text{OOS}}$.

``` python
combined = forecasts[predictor_names].________(axis=1)
mse_combined = np.mean((forecasts["actual"] - combined) ** 2)
print(f"R2_OOS {1 - mse_combined / mse_mean:.2%}")
```

## Averaging Wins, Modestly

In [ ]:
combined = forecasts[predictor_names].mean(axis=1)   # across the 12 columns
mse_combined = np.mean((forecasts["actual"] - combined) ** 2)
mse_mean = np.mean((forecasts["actual"] - forecasts["historical_mean"]) ** 2)
print(f"R2_OOS of the average: {1 - mse_combined / mse_mean:.2%}")
# The same score from each later start month, 1970-01 to 2020-12 ("MS": month start).
start_months = pd.date_range("1970-01-01", "2020-12-01", freq="MS")
wins = 0
for start in start_months:
    later = forecasts.loc[start:]            # the months from start on
    mse_combined_later = np.mean((later["actual"] - combined.loc[start:]) ** 2)
    mse_mean_later = np.mean((later["actual"] - later["historical_mean"]) ** 2)
    if mse_combined_later < mse_mean_later:
        wins = wins + 1
print(f"beats the mean from {wins} of {len(start_months)} start months")

- Small and positive, although most of its 12 parts lose. Their errors
  partly cancel.
- Better than the analysts’ average score, as it must be. Here, also
  better than each of the twelve alone.
- Caveat: scored from most later start months (1971, …, 2020), it loses.
  Start dates again.
- For the premium: Rapach, Strauss & Zhou (2010).

## Check: Average Two Forecasts

- Two forecasts: 2% and 8%. The premium turns out 3%.
- **Fingers up:** the squared error of their averaged forecast, in
  percent squared, is:
  1.  4
  2.  13
  3.  1
  4.  25

# Why Machine Learning

## Why Financial Prediction Is Hard

- **Little signal, lots of noise:** monthly, the premium’s sd is several
  times its mean. A monthly out-of-sample $R^2$ near 1% is a success.
- **Relations drift:** the verdict moved with the start year and the end
  year.
- **Many predictors, short samples:** all twelve at once did far worse
  than the historical mean on annual data.
- **Slow predictors:** neighboring windows share years, so there are few
  separate observations.

## Too Many Predictors: Five Ways Out

- **Restrict:** let finance rule out nonsense (“Use What You Know”).
- **Select:** keep a few. Each of the 12 predictors is in or out, so
  there are 2 × 2 × … × 2 subsets, each refitted every month.

<!-- -->

- **Shrink:** pull every slope toward zero. Meetings 5 and 6.
- **Summarize:** a few summary indicators (factors) built from many
  predictors. Meetings 9 and 10.
- **Average:** combine many forecasts (“Average the Forecasts”).

## What Machine Learning Brings

In [ ]:
# A preview of Meetings 7 and 8, on made-up data: the truth bends; a straight line cannot.
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import GradientBoostingRegressor
generator = np.random.default_rng(2026)
x = generator.uniform(0, 1, 400)

def truth(x):
    return np.where(x < 0.6, x, 0.6 - 1.5 * (x - 0.6))    # rises, then falls

y = truth(x) + generator.normal(0, 0.15, 400)             # plus noise
line = LinearRegression().fit(x.reshape(-1, 1), y)         # sklearn wants 2 dimensions
trees = GradientBoostingRegressor(n_estimators=200, max_depth=2, learning_rate=0.05,
                                  min_samples_leaf=20, random_state=0).fit(x.reshape(-1, 1), y)
grid = np.linspace(0, 1, 300)
plt.figure(figsize=(9, 3.0))
plt.scatter(x, y, s=8, color="0.75")
plt.plot(grid, truth(grid), color="black", linewidth=3)
plt.plot(grid, line.predict(grid.reshape(-1, 1)), color="tab:blue", linewidth=3)
plt.plot(grid, trees.predict(grid.reshape(-1, 1)), color="tab:orange", linewidth=3)
plt.title("black: truth   blue: straight line   orange: 200 small trees", fontsize=17)
plt.xlabel("predictor (simulated)")
plt.ylabel("outcome")
plt.show()

- **Shrinkage and validation:** ridge, lasso, cross-validation. Meetings
  5 and 6.
- **Curves and interactions:** trees, forests, neural networks. Meetings
  7 and 8.
- **Many assets at once:** factor models. Meetings 9 and 10.
- Not magic: disciplined prediction with many predictors and little
  signal.

## Your turn

On a card, before you leave. No names needed.

1.  A colleague adds 30 predictors and the in-sample $R^2$ rises from 1%
    to 4%. What do you expect out of sample, and why?
2.  Ten of twelve forecasts lose to the historical mean. How can their
    average beat it?

## Readings

**Required** (on Brightspace):

- Cochrane (2011), *Presidential Address: Discount Rates*, Journal of
  Finance 66(4), 1047–1108. Section I.
- Goyal & Welch (2008), *A Comprehensive Look at the Empirical
  Performance of Equity Premium Prediction*, Review of Financial Studies
  21(4), 1455–1508. Not needed for PS1.

**Optional:**

- Campbell & Thompson (2008), *Predicting Excess Stock Returns Out of
  Sample: Can Anything Beat the Historical Average?*, Review of
  Financial Studies 21(4), 1509–1531.
- Cochrane (2008), *The Dog That Did Not Bark: A Defense of Return
  Predictability*, Review of Financial Studies 21(4), 1533–1575.
- Shiller (1981), *Do Stock Prices Move Too Much to Be Justified by
  Subsequent Changes in Dividends?*, American Economic Review 71(3),
  421–436.